# Preprocessing, grouped split, encoding, dan EDA train

**Aqil — L0124139.** Melanjutkan audit tahap 1-3. Raw tetap. Semua data disimpan; ID/target dipisahkan dari X. Script `preprocess_dataset.py` dan `audit_dataset.py` harus tetap di folder yang sama. Dependensi tercatat di requirements_preprocessing.txt. Notebook tersimpan dieksekusi menggunakan Python secara berurutan, bukan kernel Jupyter pada lingkungan pembuat.

In [1]:
import importlib.util
from pathlib import Path
base = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preprocess_dataset.py').is_file()), None)
if base is None:
    base = next((p / 'kredit' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'kredit' / 'preprocess_dataset.py').is_file()), None)
if base is None:
    raise FileNotFoundError('Jalankan dari folder kredit atau root proyek.')
spec = importlib.util.spec_from_file_location('credit_preprocessing', base / 'preprocess_dataset.py')
p = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p)
print('Modul preprocessing siap.')


Modul preprocessing siap.


## 1. Pemetaan deterministik dan grup profil

Kategori dipetakan berdasarkan kebijakan audit yang sudah ditetapkan. Kelompok dibentuk dari kesamaan X setelah mapping, tanpa ID/target. Belum ada fit statistik.

In [2]:
ctx = p.load_and_recode()
print(ctx['recode_table'].to_string(index=False))


Retained 30,000 rows and 23 semantic predictors. ID excluded; target separated.
EDUCATION: 345 rows in unknown; 331 numeric codes changed.
Profiles after recoding: 29,944 groups; 108 rows in 52 repeated groups.
 code  before  after   meaning_after
    0      14    345         unknown
    1   10585  10585 graduate school
    2   14030  14030      university
    3    4917   4917     high school
    4     123    123          others
    5     280      0   merged into 0
    6      51      0   merged into 0


## 2. Split tetap

Lima fold StratifiedGroupKFold dengan seed 20261020. Fold 0=test, 1=validation, 2/3/4=train. Rasio mendekati 60/20/20 karena grup tidak dipecah. Tidak memilih split dari performa model.

In [3]:
split_summary = p.split_data(ctx)


     split  n_rows  share_percent  nondefault  default  default_percent  default_rate_difference_pp  n_groups
     train   18005         60.017       14054     3951           21.944                      -0.176     17967
validation    5997         19.990        4631     1366           22.778                       0.658      5989
      test    5998         19.993        4679     1319           21.991                      -0.129      5988
Profile-group overlap between splits: 0.


## 3. Encoding dari train saja

14 numerik dipertahankan, 9 kategori di-one-hot tanpa drop kategori acuan. Kategori baru held-out diabaikan dan dicatat, tidak dipakai fit ulang. X/y/metadata memiliki urutan baris sama.

In [4]:
schema = p.encode_data(ctx)


Encoder fit on train only: 14 numeric + 75 indicator columns = 89 model inputs.
Unseen categories per split:
split
test          0
train         0
validation    0


## 4. Korelasi train dan gambar paper

Pearson dan Spearman hanya memakai train. Fitur yang berkorelasi tidak dihapus karena merupakan objek penelitian. Korelasi bukan bukti sebab-akibat atau kontribusi model.

In [5]:
correlation_pairs = p.correlations_and_figures(ctx)


Training correlation matrices saved; no validation/test correlations used for feature choice.
feature_a feature_b  pearson_train  spearman_train
BILL_AMT1 BILL_AMT2         0.9501          0.9091
BILL_AMT1 BILL_AMT3         0.8856          0.8543
BILL_AMT1 BILL_AMT4         0.8566          0.8076
BILL_AMT1 BILL_AMT5         0.8277          0.7677
BILL_AMT1 BILL_AMT6         0.7963          0.7324
BILL_AMT2 BILL_AMT3         0.9235          0.9059
BILL_AMT2 BILL_AMT4         0.8886          0.8491
BILL_AMT2 BILL_AMT5         0.8571          0.8035
BILL_AMT2 BILL_AMT6         0.8263          0.7628
BILL_AMT3 BILL_AMT4         0.9136          0.9049
BILL_AMT3 BILL_AMT5         0.8718          0.8491
BILL_AMT3 BILL_AMT6         0.8441          0.8053
BILL_AMT4 BILL_AMT5         0.9386          0.9044
BILL_AMT4 BILL_AMT6         0.8974          0.8495
BILL_AMT5 BILL_AMT6         0.9463          0.9041


## 5. Periksa artefak hasil simpan

Periksa checksum, coverage, grup, kategori train, urutan fitur, dan label berdasarkan ID. Tidak ada model dilatih pada notebook ini.

In [6]:
checks = p.verify_outputs(ctx)
for name, passed in checks.items():
    print(name, ':', passed)


Verification passed: 30/30 checks.
raw_original_unchanged : True
raw_snapshot_unchanged : True
no_rows_dropped : True
ID_not_in_predictors : True
target_not_in_predictors : True
all_six_billing_features_retained : True
raw_billing_values_unchanged : True
negative_bill_count_unchanged : True
profile_groups_disjoint : True
ID_unique_and_covered_once : True
both_classes_in_all_splits : True
stratification_deviation_within_2pp : True
consistent_encoded_feature_order : True
encoder_categories_from_train_only : True
saved_transformer_matches : True
train_feature_names : True
train_encoded_matches_semantic : True
train_labels_match_raw_ID : True
train_row_alignment : True
train_finite : True
validation_feature_names : True
validation_encoded_matches_semantic : True
validation_labels_match_raw_ID : True
validation_row_alignment : True
validation_finite : True
test_feature_names : True
test_encoded_matches_semantic : True
test_labels_match_raw_ID : True
test_row_alignment : True
test_finite : T

## 6. Teks metode dan handoff

Masukkan teks metode dan gambar ke naskah bersama. Data encoded diserahkan ke Irfan, feature map ke Yahya. Tidak semua file teknis perlu masuk badan paper.

In [7]:
p.write_deliverables(ctx)


Paper text, tables, figures, and handoff guide saved.
